# Proctoring Analysis Review Notebook

Load results from `analyze_exam_snapshots.py` and explore per-student risk timelines,
incident summaries, and detection correlations.

**Instructions**:
1. Set `RESULTS_DIR` below to point to your output directory.
2. Run all cells (Kernel → Restart & Run All).
3. Use the dropdowns to filter by course or student.

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
RESULTS_DIR = "./results"      # folder produced by analyze_exam_snapshots.py
SUSPICIOUS_THRESHOLD = 30.0    # frames with risk_score >= this are highlighted

import json
import warnings
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 120

results_path = Path(RESULTS_DIR)
print(f"Results directory: {results_path.resolve()}")

In [ ]:
# ── Load image-level results ─────────────────────────────────────────────────
jsonl_path = results_path / "image_level_results.jsonl"

def load_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open(encoding="utf-8") as fh:
        for line in fh:
            line = line.strip()
            if not line:
                continue
            d = json.loads(line)
            flat = {
                "image_path":       d["image_path"],
                "student_id":       d["student_id"],
                "attempt_id":       d["attempt_id"],
                "timestamp":        pd.to_datetime(d["timestamp"], errors="coerce"),
                "course_id":        d["course_id"],
                "face_count":       d["face"]["count"],
                "look_away_flag":   d["attention"]["look_away_flag"],
                "severity":         d["attention"]["severity"],
                "yaw":              d["attention"].get("yaw"),
                "pitch":            d["attention"].get("pitch"),
                "identity_mismatch":d["identity"]["mismatch_flag"],
                "identity_similarity": d["identity"]["similarity_score"],
                "phone_detected":   d["objects"]["phone_detected"],
                "extra_person":     d["objects"]["extra_person_detected"],
                "book_detected":    d["objects"]["book_detected"],
                "face_obstructed":  d["objects"]["face_obstructed"],
                "blur_score":       d["quality"]["blur_score"],
                "brightness":       d["quality"]["brightness_score"],
                "low_quality":      d["quality"]["low_quality_flag"],
                "risk_score":       d["risk"]["score"],
                "reasons":          d["risk"]["reasons"],
                "error":            d.get("error"),
            }
            rows.append(flat)
    return pd.DataFrame(rows)

df = load_jsonl(jsonl_path)
print(f"Loaded {len(df):,} image-level records")
df.head()

In [ ]:
# ── Load student summary ──────────────────────────────────────────────────────
summary_path = results_path / "student_summary.csv"

summary_df = pd.read_csv(summary_path)

# Parse JSON columns
for col in ["top_reasons", "flagged_timeline"]:
    if col in summary_df.columns:
        summary_df[col] = summary_df[col].apply(
            lambda x: json.loads(x) if isinstance(x, str) else []
        )

print(f"Loaded {len(summary_df):,} student summaries")
summary_df

In [ ]:
# ── Summary table: students sorted by risk ────────────────────────────────────
RISK_ORDER = {"high": 2, "medium": 1, "low": 0}
summary_df["_risk_ord"] = summary_df["overall_risk_level"].map(RISK_ORDER).fillna(0)

display_cols = [
    "student_id", "attempt_id", "course_id",
    "total_frames", "valid_frames", "suspicious_frames", "percentage_suspicious",
    "max_risk_score", "mean_risk_score", "incident_count",
    "identity_stability_score", "overall_risk_level",
]

risk_palette = {"high": "#ffcccc", "medium": "#fff3cc", "low": "#ccffcc"}

styled = (
    summary_df.sort_values(["_risk_ord", "max_risk_score"], ascending=[False, False])
    [display_cols]
    .style
    .apply(
        lambda col: col.map(
            lambda v: f"background-color: {risk_palette.get(v, '')}" if col.name == "overall_risk_level" else ""
        ),
        axis=0,
    )
    .format({
        "percentage_suspicious": "{:.1f}%",
        "max_risk_score": "{:.1f}",
        "mean_risk_score": "{:.1f}",
        "identity_stability_score": "{:.3f}",
    })
)
styled

In [ ]:
# ── Per-student risk timeline (with interactive dropdown) ─────────────────────
try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except ImportError:
    HAS_WIDGETS = False
    print("ipywidgets not installed – run: pip install ipywidgets")
    print("Plotting first student instead.")


def plot_student_timeline(student_id: str, attempt_id: str = None) -> None:
    mask = df["student_id"] == student_id
    if attempt_id and attempt_id != "All":
        mask &= df["attempt_id"] == attempt_id
    sub = df[mask].sort_values("timestamp")

    if sub.empty:
        print(f"No data for {student_id}")
        return

    fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
    fig.suptitle(f"Student: {student_id}", fontsize=14, fontweight='bold')

    ts = sub["timestamp"]

    # ---- Risk score ----
    ax0 = axes[0]
    ax0.plot(ts, sub["risk_score"], color="#c0392b", linewidth=1.5, label="Risk Score")
    ax0.axhline(SUSPICIOUS_THRESHOLD, color="orange", linestyle="--", linewidth=1, label=f"Threshold ({SUSPICIOUS_THRESHOLD})")  
    ax0.fill_between(ts, sub["risk_score"], SUSPICIOUS_THRESHOLD,
                     where=sub["risk_score"] >= SUSPICIOUS_THRESHOLD,
                     alpha=0.25, color="red")
    ax0.set_ylabel("Risk Score")
    ax0.set_ylim(0, 105)
    ax0.legend(loc="upper right", fontsize=8)
    ax0.grid(True, alpha=0.3)

    # ---- Head yaw ----
    ax1 = axes[1]
    if sub["yaw"].notna().any():
        ax1.plot(ts, sub["yaw"], color="#2980b9", linewidth=1.2, label="Yaw")
        ax1.axhline(20, color="orange", linestyle=":", linewidth=0.8)
        ax1.axhline(-20, color="orange", linestyle=":", linewidth=0.8)
        ax1.axhline(35, color="red", linestyle=":", linewidth=0.8)
        ax1.axhline(-35, color="red", linestyle=":", linewidth=0.8)
    ax1.set_ylabel("Yaw (deg)")
    ax1.legend(loc="upper right", fontsize=8)
    ax1.grid(True, alpha=0.3)

    # ---- Event flags ----
    ax2 = axes[2]
    events = {
        "Phone":        ("phone_detected",   0.8, "purple"),
        "No Face":      (sub["face_count"] == 0, 0.6, "black"),
        "Multi Face":   (sub["face_count"] > 1,  0.4, "orange"),
        "Id Mismatch":  ("identity_mismatch",  0.2, "red"),
    }
    patches = []
    for label, (col, y_val, colour) in events.items():
        if isinstance(col, str):
            mask_ev = sub[col] == True
        else:
            mask_ev = col
        ev_ts = sub[mask_ev]["timestamp"]
        if not ev_ts.empty:
            ax2.scatter(ev_ts, [y_val] * len(ev_ts), color=colour, s=30, zorder=5)
        patches.append(mpatches.Patch(color=colour, label=label))
    ax2.set_yticks([0.2, 0.4, 0.6, 0.8])
    ax2.set_yticklabels(["Id Mismatch", "Multi Face", "No Face", "Phone"], fontsize=8)
    ax2.set_ylabel("Events")
    ax2.set_ylim(0, 1)
    ax2.grid(True, alpha=0.3)
    ax2.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M:%S"))
    ax2.set_xlabel("Timestamp")
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.show()


if HAS_WIDGETS:
    student_options = sorted(df["student_id"].unique().tolist())
    attempt_options = ["All"] + sorted(df["attempt_id"].unique().tolist())

    student_dd = widgets.Dropdown(options=student_options, description="Student:", layout=widgets.Layout(width='300px'))
    attempt_dd = widgets.Dropdown(options=attempt_options, description="Attempt:", layout=widgets.Layout(width='300px'))

    out = widgets.interactive_output(
        plot_student_timeline,
        {"student_id": student_dd, "attempt_id": attempt_dd}
    )
    display(widgets.HBox([student_dd, attempt_dd]), out)
else:
    first_student = df["student_id"].iloc[0]
    plot_student_timeline(first_student)

In [ ]:
# ── Detection event correlation heatmap ──────────────────────────────────────
numeric_events = [
    "face_count", "look_away_flag", "identity_mismatch",
    "phone_detected", "extra_person", "book_detected",
    "face_obstructed", "low_quality", "risk_score",
]

bool_cols = ["look_away_flag", "identity_mismatch", "phone_detected",
             "extra_person", "book_detected", "face_obstructed", "low_quality"]

corr_df = df[numeric_events].copy()
for c in bool_cols:
    corr_df[c] = corr_df[c].astype(float)

corr = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdYlGn', vmin=-1, vmax=1)
fig.colorbar(im, ax=ax, shrink=0.8)
labels = [c.replace('_', '\n') for c in numeric_events]
ax.set_xticks(range(len(numeric_events)))
ax.set_yticks(range(len(numeric_events)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_yticklabels(labels, fontsize=8)
for i in range(len(numeric_events)):
    for j in range(len(numeric_events)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha='center', va='center', fontsize=7)
ax.set_title("Detection Event Correlation Matrix", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Course-level overview ─────────────────────────────────────────────────────
course_stats = (
    df.groupby("course_id")
    .agg(
        total_frames=("risk_score", "count"),
        mean_risk=("risk_score", "mean"),
        max_risk=("risk_score", "max"),
        suspicious_frames=("risk_score", lambda x: (x >= SUSPICIOUS_THRESHOLD).sum()),
        phone_events=("phone_detected", "sum"),
        no_face_events=("face_count", lambda x: (x == 0).sum()),
    )
    .reset_index()
)
course_stats["suspicious_pct"] = (
    course_stats["suspicious_frames"] / course_stats["total_frames"] * 100
).round(1)

print("Course-level summary:")
display(course_stats.sort_values("mean_risk", ascending=False))

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(course_stats["course_id"], course_stats["mean_risk"], color="steelblue", alpha=0.8)
ax.set_xlabel("Course")
ax.set_ylabel("Mean Risk Score")
ax.set_title("Mean Risk Score per Course")
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# ── Incident details for high-risk students ───────────────────────────────────
high_risk = summary_df[summary_df["overall_risk_level"] == "high"].sort_values(
    "max_risk_score", ascending=False
)

if high_risk.empty:
    print("No high-risk students found.")
else:
    print(f"High-risk students: {len(high_risk)}")
    for _, row in high_risk.iterrows():
        print(f"\n{'='*60}")
        print(f"Student: {row['student_id']}  |  Attempt: {row['attempt_id']}  |  Course: {row['course_id']}")
        print(f"  Frames: {row['valid_frames']} valid, {row['suspicious_frames']} suspicious ({row['percentage_suspicious']:.1f}%)")
        print(f"  Max risk: {row['max_risk_score']:.0f}  |  Mean risk: {row['mean_risk_score']:.1f}")
        print(f"  Incidents: {row['incident_count']}")
        print(f"  Top reasons: {row['top_reasons']}")
        print(f"  Identity stability: {row['identity_stability_score']:.3f}")

        for event in row["flagged_timeline"][:5]:
            print(f"    [{event['timestamp']}]  risk={event['risk_score']:.0f}  reasons={event['reasons']}")
        if len(row["flagged_timeline"]) > 5:
            print(f"    ... and {len(row['flagged_timeline']) - 5} more flagged frames")

In [ ]:
# ── Quality distribution ──────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(df["blur_score"].dropna(), bins=40, color='steelblue', edgecolor='white')
axes[0].set_title("Blur Score Distribution"); axes[0].set_xlabel("Laplacian Variance")

axes[1].hist(df["brightness"].dropna(), bins=40, color='goldenrod', edgecolor='white')
axes[1].set_title("Brightness Distribution"); axes[1].set_xlabel("Mean Pixel Value")

axes[2].hist(df["risk_score"].dropna(), bins=20, color='tomato', edgecolor='white')
axes[2].axvline(SUSPICIOUS_THRESHOLD, color='black', linestyle='--', label=f'Threshold {SUSPICIOUS_THRESHOLD}')
axes[2].set_title("Risk Score Distribution"); axes[2].set_xlabel("Risk Score")
axes[2].legend(fontsize=8)

plt.tight_layout()
plt.show()